# Two-stream MLP with a learned per-log gate

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from ynbtriage.model.features import make_featurizer
from ynbtriage.model.pipeline import fit_baselines
from ynbtriage.model.train import TrainConfig

import two_stream as ts   # experiments/two_stream.py

#DB_PATH = os.environ.get("YNB_DB_PATH")
DB_PATH = "../ynobuild-final-annotations.db"
SEEDS = (7400, 7401, 7402, 7403, 7404)
ANALYSIS_SPLIT = "test"   # gate/contribution/knockout analysis
CFG = TrainConfig()      # preserve original settings: Adam 1e-3, wd 1e-4, dropout 0.2, batch 32, patience 25

# colors: same pair as the repo's loss curves
C_ERR, C_TAIL = "#2a78d6", "#eb6834"
print("DB:", DB_PATH)

DB: ../ynobuild-final-annotations.db


## 1. Data: same rows, splits and labels as original, plus the split excerpt

In [2]:
df, all_labels = ts.load_two_stream(DB_PATH)
df, labels, dropped, parts, y, texts = ts.prepare(df, all_labels)   # REUSED from pipeline.py

print("labels:", labels)
print({s: len(d) for s, d in parts.items()})
print("\nShare of logs where the regex found an error line:")
print(df.groupby("split")["error_line_found"].mean().round(3).to_string())

labels: ['environment_decay', 'recipe_error', 'build_execution']
{'train': 522, 'val': 156, 'test': 156}

Share of logs where the regex found an error line:
split
test     0.994
train    0.994
val      0.987


## 2. Features

- **Baseline:** one `TfidfFeaturizer(dim=256)` over the merged excerpt. This is the original final model's input.
- **Two-stream:** one `TfidfFeaturizer(dim=128)` per region, fit on train only, with the outputs placed side by side. The total input size stays at 256, so the two models get the same input budget.

In [3]:
feat_merged = make_featurizer("tfidf", tfidf_dim=256).fit(texts["train"])
X_merged = {s: feat_merged.transform(t) for s, t in texts.items()}

feat_two = ts.TwoStreamFeaturizer(dim_per_region=128).fit(parts["train"]["err_text"],
                                                           parts["train"]["tail_text"])
X_two = {s: feat_two.transform(d["err_text"], d["tail_text"]) for s, d in parts.items()}
d_err = feat_two.d_err
print("merged:", X_merged["train"].shape, "| two-stream:", X_two["train"].shape, "| d_err =", d_err)

merged: (522, 256) | two-stream: (522, 256) | d_err = 128


## 3. Train both models across seeds and evaluate accuracy

Both models use the same `TrainConfig`, the same seeds and the same `train_mlp` loop. The only differences are the input layout and the architecture.

In [4]:
runs = ts.run_seeds(X_merged, X_two, y, d_err, labels, seeds=SEEDS, cfg=CFG)

seed 7400: macro-F1 val  MLP 0.852  two-stream 0.870   (epochs 40, 42)
seed 7401: macro-F1 val  MLP 0.858  two-stream 0.858   (epochs 40, 42)
seed 7402: macro-F1 val  MLP 0.858  two-stream 0.870   (epochs 39, 43)
seed 7403: macro-F1 val  MLP 0.858  two-stream 0.858   (epochs 40, 48)
seed 7404: macro-F1 val  MLP 0.858  two-stream 0.858   (epochs 39, 43)


In [5]:
for split in ("val", "test"):
    print(f"--- {split} (mean ± sd over {len(SEEDS)} seeds) ---")
    print(ts.summarize_runs(runs, split).round(3).to_string(index=False), "\n")

# per-class F1, the place a side effect would show
pcf = pd.DataFrame({
    m: pd.DataFrame([r[f"{k}_{ANALYSIS_SPLIT}"]["per_class_f1"] for r in runs]).mean()
    for m, k in (("Original MLP", "mlp"), ("Two-stream", "two"))})
print(f"per-class F1 ({ANALYSIS_SPLIT}, mean over seeds)")
pcf.round(3)

--- val (mean ± sd over 5 seeds) ---
                        model  accuracy_mean  accuracy_sd  macro_f1_mean  macro_f1_sd
Original MLP (merged excerpt)          0.851        0.003          0.857        0.003
             Two-stream gated          0.858        0.007          0.863        0.007 

--- test (mean ± sd over 5 seeds) ---
                        model  accuracy_mean  accuracy_sd  macro_f1_mean  macro_f1_sd
Original MLP (merged excerpt)          0.924        0.005          0.925        0.006
             Two-stream gated          0.924        0.007          0.926        0.007 

per-class F1 (test, mean over seeds)


,Original MLP,Two-stream
environment_decay,0.927,0.928
recipe_error,0.945,0.955
build_execution,0.903,0.894


## 4. Look at contribution of each log excerpt component

For every log in the analysis split, and for every seed:

- `w_err`: the gate's weight on the error-window branch. The log tail gets `1 - w_err`.
- `contrib_err`: the share of the fused vector's size that actually comes from the error window. This catches cases where a small weight multiplies a large branch output.

In [6]:
meas = pd.concat([ts.per_log_measurements(r["two"], X_two[ANALYSIS_SPLIT], y[ANALYSIS_SPLIT], labels)
                  .assign(seed=r["seed"]) for r in runs], ignore_index=True)

by_class = (meas.groupby("true")[["w_err", "contrib_err"]].mean()
            .reindex(labels).rename(columns={"w_err": "gate weight on error window",
                                             "contrib_err": "contribution share of error window"}))
by_class.round(3)

,gate weight on error window,contribution share of error window
true,,
environment_decay,0.265,0.257
recipe_error,0.432,0.432
build_execution,0.407,0.405


The most comprehensive of the three measurements is the "knockout" analysis. If removing a branch barely changes accuracy, that branch wasn't doing much (regardless of the gate weight or contribution share).

In [7]:
ko = (pd.concat([ts.knockout_summary(g).assign(seed=s) for s, g in meas.groupby("seed")])
      .groupby("variant", sort=False)[["accuracy", "flip_rate_vs_full"]].agg(["mean", "std"]))
ko.round(3)

accuracy        flip_rate_vs_full       
                         mean    std              mean    std
variant                                                      
full model              0.924  0.007             0.000  0.000
error window removed    0.918  0.005             0.041  0.016
tail removed            0.860  0.015             0.114  0.019

In [8]:
# Same knockout, per true class: which class depends on which region?
per_class_ko = meas.assign(
    full=meas["pred"] == meas["true"],
    no_err_window=meas["pred_ko_err"] == meas["true"],
    no_tail=meas["pred_ko_tail"] == meas["true"],
).groupby("true")[["full", "no_err_window", "no_tail"]].mean().reindex(labels)
per_class_ko.round(3)

,full,no_err_window,no_tail
true,,,
environment_decay,0.918,0.929,0.864
recipe_error,0.933,0.933,0.883
build_execution,0.923,0.892,0.835
